# Report Mode — Integration Test

Generates a full structured case review report for a single case.

Report structure follows the credit_risk pillar format:
1. Why and how the default occurred?
2. Early risk indicators (>6m before default)
3. How did modeling scores react?
4. Major behavioral changes towards default (<6m)

Followed by per-specialist domain sections and cross-domain synthesis.

In [ ]:
%load_ext dotenv
%dotenv

%load_ext autoreload
%autoreload 2

import os, sys, json
import nest_asyncio
nest_asyncio.apply()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'simulated')
PROFILE_DIR = os.path.join(PROJECT_ROOT, 'config', 'data_profiles')
PILLAR_DIR = os.path.join(PROJECT_ROOT, 'config', 'pillars')

# ═══════════════ CONFIG ═══════════════
SAFECHAIN_MODEL = "gpt-4.1"
CASE_ID = "CASE-00001"
PILLAR = "credit_risk"
MODE = "report"
QUESTION = "Generate a full credit risk case review report for this customer."
# ═════════════════════════════════════

print(f'Case: {CASE_ID} | Pillar: {PILLAR} | Mode: {MODE}')
print(f'Question: {QUESTION}')

## 1. Load Data

In [ ]:
from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools

gw = SimulatedDataGateway.from_case_folders(DATA_DIR)
catalog = DataCatalog(profile_dir=PROFILE_DIR)
init_tools(gw, catalog)
gw.set_case(CASE_ID)

print(f'Tables: {gw.list_tables()}')
print(f'Row counts:')
for t in gw.list_tables():
    rows = gw.query(t)
    print(f'  {t}: {len(rows) if rows else 0} rows')

## 2. Connect LLM

In [ ]:
from gateway.safechain_adapter import SafeChainAdapter

try:
    from safechain.lcel import model as safechain_model
    llm = safechain_model(SAFECHAIN_MODEL)
    adapter = SafeChainAdapter(llm=llm, model_name=SAFECHAIN_MODEL)
    print(f'SafeChain adapter: model={SAFECHAIN_MODEL}')
except ImportError:
    from gateway.openai_adapter import OpenAIAdapter
    adapter = OpenAIAdapter(model='gpt-4.1')
    print(f'OpenAI adapter (SafeChain not available)')

## 3. Initialize Pipeline

In [ ]:
from gateway.firewall_stack import FirewallStack
from log.event_logger import EventLogger
from agents.session_registry import SessionRegistry
from agents.general_specialist import GeneralSpecialist
from orchestrator.orchestrator import Orchestrator
from orchestrator.team import TeamConstructor
from orchestrator.chat_agent import ChatAgent
from config.pillar_loader import PillarLoader
from skills.domain.loader import load_domain_skill, list_domain_skills

logger = EventLogger(session_id=f'report-{CASE_ID}', log_dir=os.path.join(PROJECT_ROOT, 'logs'))
pillar_loader = PillarLoader(pillar_dir=PILLAR_DIR)
pillar_config = pillar_loader.load(PILLAR) or {}
registry = SessionRegistry()

logger.log('session_start', {'case_id': CASE_ID, 'pillar': PILLAR, 'mode': MODE})
print(f'Available specialists: {list_domain_skills()}')
print(f'Pillar report_format: {len(pillar_config.get("report_format", ""))} chars')
print(f'Pillar synthesis_report sections: {list(pillar_config.get("synthesis_report", {}).keys())}')

## 4. Team Construction

For report mode, typically all specialists are selected.

In [ ]:
logger.set_trace('report-001')
team_fw = FirewallStack(adapter=adapter, logger=logger)
team_constructor = TeamConstructor(firewall=team_fw, logger=logger)

selected = team_constructor.select_specialists(
    question=QUESTION, pillar=PILLAR,
    available_specialists=list_domain_skills(),
    active_specialists=[],
)
print(f'Selected specialists ({len(selected)}): {selected}')

## 5. Run All Specialists (Report Mode)

Each specialist produces a structured section following the pillar's `report_instructions`.

In [ ]:
specialist_outputs = {}

for domain in selected:
    skill = load_domain_skill(domain)
    if skill is None:
        print(f'SKIP {domain}: skill not found')
        continue
    
    spec_config = pillar_loader.get_specialist_config(PILLAR, domain) or {}
    fw = FirewallStack(adapter=adapter, logger=logger)
    agent = registry.get_or_create(domain, PILLAR, skill, spec_config, fw, logger)
    
    print(f'Running {domain}...', end=' ')
    output = agent.run(QUESTION, mode=MODE)
    specialist_outputs[domain] = output
    print(f'done ({len(output.findings)} chars)')

print(f'\nAll specialists completed: {list(specialist_outputs.keys())}')

## 6. Inspect Individual Specialist Reports

In [ ]:
for domain, output in specialist_outputs.items():
    print(f'\n{"="*60}')
    print(f'SPECIALIST: {domain.upper()}')
    print(f'{"="*60}')
    print(f'\nFindings:\n{output.findings[:1000]}')
    if len(output.findings) > 1000:
        print(f'... ({len(output.findings) - 1000} more chars)')
    print(f'\nEvidence ({len(output.evidence)} items): {output.evidence[:3]}')
    print(f'Implications ({len(output.implications)} items): {output.implications[:3]}')
    print(f'Data gaps: {output.data_gaps}')

## 7. General Specialist — Compare

In [ ]:
compare_fw = FirewallStack(adapter=adapter, logger=logger)
general = GeneralSpecialist(firewall=compare_fw, logger=logger)
review_report = general.compare(specialist_outputs, QUESTION)

print(f'Resolved contradictions: {len(review_report.resolved)}')
for r in review_report.resolved:
    print(f'  {r.pair}: {r.contradiction[:100]}')
    print(f'  → {r.conclusion[:150]}')

print(f'\nOpen conflicts: {len(review_report.open_conflicts)}')
for c in review_report.open_conflicts:
    print(f'  {c.pair}: {c.reason_unresolved[:150]}')

print(f'\nCross-domain insights ({len(review_report.cross_domain_insights)}):')
for ins in review_report.cross_domain_insights:
    print(f'  - {ins}')

## 8. Orchestrator Synthesis (Report Mode)

Assembles the final report using:
- `report.yaml` base synthesis rules
- `credit_risk.yaml` report_format (section order)
- `credit_risk.yaml` synthesis_report (overview sections + section-specific prompts)

In [ ]:
synth_fw = FirewallStack(adapter=adapter, logger=logger)
orchestrator = Orchestrator(synth_fw, logger, registry, PILLAR, pillar_config=pillar_config)

print('Synthesizing report...', end=' ')
final = orchestrator.synthesize(specialist_outputs, review_report, QUESTION, MODE)
print(f'done ({len(final.answer)} chars)')

print(f'\nResolved contradictions: {len(final.resolved_contradictions)}')
print(f'Open conflicts: {len(final.open_conflicts)}')
print(f'Data gaps: {len(final.data_gaps)}')
print(f'Blocked steps: {len(final.blocked_steps)}')
print(f'Specialists consulted: {final.specialists_consulted}')

## 9. Final Report

In [ ]:
chat_agent = ChatAgent(firewall=synth_fw, logger=logger)
formatted = chat_agent.format_for_reviewer(final)

print('='*80)
print(f'CREDIT RISK CASE REVIEW — {CASE_ID}')
print(f'Pillar: {PILLAR} | Mode: {MODE}')
print('='*80)
print(formatted)
print('='*80)

## 10. Render Report as Markdown

If running in Jupyter, render the report answer as formatted markdown.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(final.answer))

## 11. Data Gaps & Open Items

In [ ]:
if final.data_gaps:
    print('DATA GAPS:')
    for g in final.data_gaps:
        signal = ' *** SIGNAL ***' if g.is_signal else ''
        print(f'  [{g.specialist}] {g.missing_data}{signal}')
        print(f'    Interpretation: {g.absence_interpretation}')
else:
    print('No data gaps detected.')

print()

if final.open_conflicts:
    print('OPEN CONFLICTS (require human judgment):')
    for c in final.open_conflicts:
        print(f'  {c.pair[0]} vs {c.pair[1]}: {c.contradiction}')
        print(f'    Why unresolved: {c.reason_unresolved}')
else:
    print('No open conflicts.')

if final.blocked_steps:
    print(f'\nBLOCKED ANALYSES: {len(final.blocked_steps)}')
    for b in final.blocked_steps:
        print(f'  [{b.specialist}] {b.error}')

## 12. Inspect Event Log

In [ ]:
log_path = os.path.join(PROJECT_ROOT, 'logs', f'report-{CASE_ID}.jsonl')

if os.path.exists(log_path):
    with open(log_path) as f:
        events = [json.loads(line) for line in f]
    
    from collections import Counter
    type_counts = Counter(e['event'] for e in events)
    print(f'Total events: {len(events)}')
    for event_type, count in type_counts.most_common():
        print(f'  {event_type}: {count}')
else:
    print(f'No log at {log_path}')

In [ ]:
logger.log('session_end', {})
print('Session complete.')